# NYC Yellow Taxi Dataset Integration

This notebook integrates the **already preprocessed** Yellow Taxi zone-hour demand, Census taxi-zone context, subway proximity, and hourly weather datasets.

The Yellow Taxi zone-hour dataset is used as the master backbone. The notebook performs only the compatibility step required for weather before carrying out the joins, followed by a simple final validation.

**Final data grain:** one row per `LocationID × pickup_hour`.


In [13]:
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)


## 1. Define File Paths

In [14]:
PROCESSED_DIR = Path(
    r"C:\Asia Pacific University\All Module Notes\Semister-5\Investigations"
    r"\FYP Semester 1\Progress\fyp\processed_outputs"
)

TAXI_FILE = (
    PROCESSED_DIR / "yellow_taxi" /
    "nyc_yellow_taxi_zone_hour_demand_2025_q1.csv"
)

CENSUS_FILE = (
    PROCESSED_DIR / "spatial_joining" /
    "nyc_taxi_zones_with_zero_vehicle_context.csv"
)

SUBWAY_FILE = (
    PROCESSED_DIR / "mta_subway" /
    "nyc_taxi_zone_subway_proximity.csv"
)

WEATHER_FILE = (
    PROCESSED_DIR / "weather" /
    "nyc_central_park_weather_hourly_2025_q1.csv"
)

MODEL_DATA_DIR = PROCESSED_DIR / "model_datasets"
MODEL_DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    MODEL_DATA_DIR /
    "nyc_yellow_taxi_master_analysis_ready_2025_q1.csv"
)

BASELINE_FILE = (
    MODEL_DATA_DIR /
    "nyc_yellow_taxi_baseline_2025_q1.csv"
)

ENRICHED_FILE = (
    MODEL_DATA_DIR /
    "nyc_yellow_taxi_enriched_2025_q1.csv"
)

for file_path in [TAXI_FILE, CENSUS_FILE, SUBWAY_FILE, WEATHER_FILE]:
    print(f"{file_path}: {'FOUND' if file_path.exists() else 'MISSING'}")

print("\nModel dataset folder:")
print(MODEL_DATA_DIR)


C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs\yellow_taxi\nyc_yellow_taxi_zone_hour_demand_2025_q1.csv: FOUND
C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs\spatial_joining\nyc_taxi_zones_with_zero_vehicle_context.csv: FOUND
C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs\mta_subway\nyc_taxi_zone_subway_proximity.csv: FOUND
C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs\weather\nyc_central_park_weather_hourly_2025_q1.csv: FOUND

Model dataset folder:
C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs\model_datasets


## 2. Load the Preprocessed Datasets

### 2.1 Integration Inputs and Master Backbone

Evidence for the dimensions of the four processed datasets used in the final integration.

In [15]:
taxi = pd.read_csv(TAXI_FILE)
census = pd.read_csv(CENSUS_FILE)
subway = pd.read_csv(SUBWAY_FILE)
weather = pd.read_csv(WEATHER_FILE)

dataset_shapes = pd.DataFrame({
    "Dataset": ["Taxi zone-hour demand", "Census taxi-zone context", "Subway proximity", "Weather"],
    "Rows": [len(taxi), len(census), len(subway), len(weather)],
    "Columns": [taxi.shape[1], census.shape[1], subway.shape[1], weather.shape[1]]
})

display(dataset_shapes)


,Dataset,Rows,Columns
0,Taxi zone-hour demand,565658,8
1,Census taxi-zone context,262,14
2,Subway proximity,262,10
3,Weather,2160,9


## 3. Prepare Weather for Integration

The weather dataset contains one additional local timestamp that is not present in the taxi zone-hour timeline. The following cells identify that row first, remove only that row, then confirm that the timestamp fields can be aligned.


### 3.1 Weather Timestamp Compatibility

Evidence for identifying and resolving the one-hour difference between the taxi and weather timelines.

In [16]:
print("Weather rows:", len(weather))
print("Taxi unique pickup-hour values:", taxi["pickup_hour"].nunique())

problem_weather_rows = weather.loc[weather["dst_nonexistent_hour"] == True].copy()

print("\nWeather rows flagged as nonexistent DST hours:", len(problem_weather_rows))
display(problem_weather_rows)


Weather rows: 2160
Taxi unique pickup-hour values: 2159

Weather rows flagged as nonexistent DST hours: 1


,timestamp,date,hour,temperature_f,precipitation_in,temperature_source,precipitation_source,temperature_was_interpolated,dst_nonexistent_hour
1610,2025-03-09 02:00:00,2025-03-09,2,34.52,0.0,time_interpolation,daily_total_supported_zero,True,True


In [17]:
weather_for_join = weather.loc[
    weather["dst_nonexistent_hour"] == False
].copy()

print("Weather rows before removal:", len(weather))
print("Weather rows after removal:", len(weather_for_join))
print("Rows removed:", len(weather) - len(weather_for_join))


Weather rows before removal: 2160
Weather rows after removal: 2159
Rows removed: 1


### 3.2 Check and Standardise the Datetime Fields

In [18]:
datetime_check = pd.DataFrame({
    "Dataset": ["Taxi", "Weather"],
    "Column": ["pickup_hour", "timestamp"],
    "Current dtype": [str(taxi["pickup_hour"].dtype), str(weather_for_join["timestamp"].dtype)],
    "Example value": [taxi["pickup_hour"].iloc[0], weather_for_join["timestamp"].iloc[0]]
})

display(datetime_check)


,Dataset,Column,Current dtype,Example value
0,Taxi,pickup_hour,object,1/1/2025 0:00
1,Weather,timestamp,object,2025-01-01 00:00:00


In [19]:
taxi["join_hour"] = pd.to_datetime(
    taxi["pickup_hour"],
    format="%d/%m/%Y %H:%M"
)

weather_for_join["join_hour"] = pd.to_datetime(
    weather_for_join["timestamp"],
    format="%Y-%m-%d %H:%M:%S"
)

converted_datetime_check = pd.DataFrame({
    "Dataset": ["Taxi", "Weather"],
    "Converted dtype": [str(taxi["join_hour"].dtype), str(weather_for_join["join_hour"].dtype)],
    "Minimum timestamp": [taxi["join_hour"].min(), weather_for_join["join_hour"].min()],
    "Maximum timestamp": [taxi["join_hour"].max(), weather_for_join["join_hour"].max()]
})

display(converted_datetime_check)


,Dataset,Converted dtype,Minimum timestamp,Maximum timestamp
0,Taxi,datetime64[ns],2025-01-01,2025-03-31 23:00:00
1,Weather,datetime64[ns],2025-01-01,2025-03-31 23:00:00


In [20]:
taxi_hours = set(taxi["join_hour"].unique())
weather_hours = set(weather_for_join["join_hour"].unique())

hour_alignment = pd.Series({
    "taxi_unique_hours": len(taxi_hours),
    "weather_unique_hours": len(weather_hours),
    "taxi_hours_without_weather": len(taxi_hours - weather_hours),
    "weather_hours_without_taxi": len(weather_hours - taxi_hours),
})

display(hour_alignment.to_frame("value"))

assert taxi_hours == weather_hours, "Taxi and weather hourly timestamps do not match."
print("Taxi and weather timestamps match exactly.")


,value
taxi_unique_hours,2159
weather_unique_hours,2159
taxi_hours_without_weather,0
weather_hours_without_taxi,0


Taxi and weather timestamps match exactly.


## 4. Join Census Context

The Census context is already aggregated to taxi-zone level, so it can be joined directly to the taxi zone-hour backbone using `LocationID`.


### 4.1 Census Context Integration

Integration of taxi-zone Census context using `LocationID` while preserving the complete taxi zone-hour backbone.

In [21]:
master = taxi.merge(
    census,
    on="LocationID",
    how="left",
    validate="many_to_one"
)

print("Rows before Census join:", len(taxi))
print("Rows after Census join:", len(master))
print("Missing zero-vehicle household rate:", master["zero_vehicle_household_rate"].isna().sum())


Rows before Census join: 565658
Rows after Census join: 565658
Missing zero-vehicle household rate: 6477


## 5. Join Subway Proximity

Only subway-specific fields are added because the Census taxi-zone file already contains the taxi-zone name and borough fields.


### 5.1 Subway Proximity Integration

Integration of the preprocessed taxi-zone subway proximity information using `LocationID`.

In [22]:
subway_for_join = subway[[
    "LocationID",
    "nearest_station_id",
    "nearest_station_name",
    "station_borough",
    "nearest_station_division",
    "nearest_station_latitude",
    "nearest_station_longitude",
    "subway_proximity_km"
]].copy()

master = master.merge(
    subway_for_join,
    on="LocationID",
    how="left",
    validate="many_to_one"
)

print("Rows after Subway join:", len(master))
print("Missing subway proximity:", master["subway_proximity_km"].isna().sum())


Rows after Subway join: 565658
Missing subway proximity: 0


## 6. Join Hourly Weather

### 6.1 Hourly Weather Integration

Integration of the aligned hourly temperature and precipitation variables using the standardised hourly timestamp.

In [23]:
weather_join = weather_for_join[[
    "join_hour",
    "temperature_f",
    "precipitation_in",
    "temperature_source",
    "precipitation_source",
    "temperature_was_interpolated"
]].copy()

master = master.merge(
    weather_join,
    on="join_hour",
    how="left",
    validate="many_to_one"
)

print("Rows after Weather join:", len(master))
print("Missing temperature:", master["temperature_f"].isna().sum())
print("Missing precipitation:", master["precipitation_in"].isna().sum())


Rows after Weather join: 565658
Missing temperature: 0
Missing precipitation: 0


## 7. Final Integration Validation

This validation checks only whether the joins preserved the expected master-table structure and whether the main joined variables are present.


### 7.1 Validation of the Complete Integrated Dataset

Validation of the complete 262-zone integrated master dataset before any analytical treatment.

In [24]:
duplicate_zone_hours = master.duplicated(
    subset=["LocationID", "join_hour"]
).sum()

validation = pd.Series({
    "rows_equal_565658": len(master) == 565_658,
    "unique_taxi_zones_equal_262": master["LocationID"].nunique() == 262,
    "unique_hours_equal_2159": master["join_hour"].nunique() == 2_159,
    "duplicate_zone_hours_equal_0": duplicate_zone_hours == 0,
    "subway_missing_equal_0": master["subway_proximity_km"].isna().sum() == 0,
    "temperature_missing_equal_0": master["temperature_f"].isna().sum() == 0,
    "precipitation_missing_equal_0": master["precipitation_in"].isna().sum() == 0,
    "pickup_total_preserved": master["pickup_count"].sum() == taxi["pickup_count"].sum(),
})

display(validation.to_frame("passed"))

print("\nKnown Census missing rows:", master["zero_vehicle_household_rate"].isna().sum())
print("Final rows:", len(master))
print("Unique taxi zones:", master["LocationID"].nunique())
print("Unique hours:", master["join_hour"].nunique())
print("Duplicate zone-hour rows:", duplicate_zone_hours)
print("Total pickup count:", int(master["pickup_count"].sum()))

assert validation.all(), "The integrated master dataset failed validation."
print("\nAll core integration checks passed.")


,passed
rows_equal_565658,True
unique_taxi_zones_equal_262,True
unique_hours_equal_2159,True
duplicate_zone_hours_equal_0,True
subway_missing_equal_0,True
temperature_missing_equal_0,True
precipitation_missing_equal_0,True
pickup_total_preserved,True



Known Census missing rows: 6477
Final rows: 565658
Unique taxi zones: 262
Unique hours: 2159
Duplicate zone-hour rows: 0
Total pickup count: 10218621

All core integration checks passed.


## 8. Preview the Master Dataset

### 8.1 Preview of the Complete Master Dataset

Preview of the complete integrated table before master-dataset inspection and treatment.

In [25]:
print("Master dataset shape:", master.shape)
print("\nMaster dataset columns:")
print(master.columns.tolist())

display(master.head())


Master dataset shape: (565658, 34)

Master dataset columns:
['LocationID', 'pickup_hour', 'date', 'hour', 'day_of_week', 'month', 'pickup_count', 'was_zero_padded', 'join_hour', 'borough', 'zone', 'service_zone', 'estimated_total_households', 'estimated_zero_vehicle_households', 'zero_vehicle_household_rate', 'zero_vehicle_rate_moe', 'zero_vehicle_rate_cv', 'census_reliability', 'contributing_tract_count', 'census_coverage_ratio', 'census_feature_available', 'census_feature_status', 'nearest_station_id', 'nearest_station_name', 'station_borough', 'nearest_station_division', 'nearest_station_latitude', 'nearest_station_longitude', 'subway_proximity_km', 'temperature_f', 'precipitation_in', 'temperature_source', 'precipitation_source', 'temperature_was_interpolated']


,LocationID,pickup_hour,date,hour,day_of_week,month,pickup_count,was_zero_padded,join_hour,borough,zone,service_zone,estimated_total_households,estimated_zero_vehicle_households,zero_vehicle_household_rate,zero_vehicle_rate_moe,zero_vehicle_rate_cv,census_reliability,contributing_tract_count,census_coverage_ratio,census_feature_available,census_feature_status,nearest_station_id,nearest_station_name,station_borough,nearest_station_division,nearest_station_latitude,nearest_station_longitude,subway_proximity_km,temperature_f,precipitation_in,temperature_source,precipitation_source,temperature_was_interpolated
0,2,1/1/2025 0:00,1/1/2025,0,2,1,0,True,2025-01-01 00:00:00,Queens,Jamaica Bay,Boro Zone,17.772228,2.108158,0.118621,1.106223,5.669621,low,5,1.0,True,available,199,Broad Channel,Q,IND,40.608382,-73.815925,1.598622,44.96,0.0,reported_accepted,reported_accepted,False
1,2,1/1/2025 1:00,1/1/2025,1,2,1,0,True,2025-01-01 01:00:00,Queens,Jamaica Bay,Boro Zone,17.772228,2.108158,0.118621,1.106223,5.669621,low,5,1.0,True,available,199,Broad Channel,Q,IND,40.608382,-73.815925,1.598622,44.96,0.0,reported_accepted,reported_accepted,False
2,2,1/1/2025 2:00,1/1/2025,2,2,1,0,True,2025-01-01 02:00:00,Queens,Jamaica Bay,Boro Zone,17.772228,2.108158,0.118621,1.106223,5.669621,low,5,1.0,True,available,199,Broad Channel,Q,IND,40.608382,-73.815925,1.598622,44.96,0.0,reported_accepted,reported_accepted,False
3,2,1/1/2025 3:00,1/1/2025,3,2,1,0,True,2025-01-01 03:00:00,Queens,Jamaica Bay,Boro Zone,17.772228,2.108158,0.118621,1.106223,5.669621,low,5,1.0,True,available,199,Broad Channel,Q,IND,40.608382,-73.815925,1.598622,46.94,0.0,reported_accepted,reported_accepted,False
4,2,1/1/2025 4:00,1/1/2025,4,2,1,0,True,2025-01-01 04:00:00,Queens,Jamaica Bay,Boro Zone,17.772228,2.108158,0.118621,1.106223,5.669621,low,5,1.0,True,available,199,Broad Channel,Q,IND,40.608382,-73.815925,1.598622,48.02,0.0,reported_accepted,reported_accepted,False


## 9. Master Dataset Inspection and Census Missing-Value Treatment

After all four processed datasets have been integrated and the complete 262-zone backbone has been validated, the master dataset is inspected for unresolved missing values before column selection and export.

The purpose of this step is to distinguish a successful join from values that are genuinely unavailable in the source data.


### 9.1 Inspection of Missing Contextual Values

Inspection of the integrated master dataset for unresolved missing values before analysis.

In [26]:
# Summarise missing values in the main variables required for later analysis.
inspection_missing = pd.Series({
    "zero_vehicle_household_rate": master["zero_vehicle_household_rate"].isna().sum(),
    "subway_proximity_km": master["subway_proximity_km"].isna().sum(),
    "temperature_f": master["temperature_f"].isna().sum(),
    "precipitation_in": master["precipitation_in"].isna().sum(),
})

display(inspection_missing.to_frame("missing_rows"))


,missing_rows
zero_vehicle_household_rate,6477
subway_proximity_km,0
temperature_f,0
precipitation_in,0


### 9.2 Investigation and Treatment of Census Missing Zones

Identification of the affected taxi zones and examination of their Census household information and Q1 Yellow Taxi demand.

In [27]:
# Identify the taxi zones responsible for the unavailable Census rate.
census_missing_zones = (
    master.loc[
        master["zero_vehicle_household_rate"].isna(),
        [
            "LocationID",
            "borough",
            "zone",
            "estimated_total_households",
            "estimated_zero_vehicle_households",
            "zero_vehicle_household_rate"
        ]
    ]
    .drop_duplicates()
    .sort_values("LocationID")
)

print("Taxi zones with unavailable zero-vehicle household rate:",
      len(census_missing_zones))
display(census_missing_zones)


Taxi zones with unavailable zero-vehicle household rate: 3


,LocationID,borough,zone,estimated_total_households,estimated_zero_vehicle_households,zero_vehicle_household_rate
218059,103,Manhattan,Governor's Island/Ellis Island/Liberty Island,0.0,0.0,NaN
220218,104,Manhattan,Governor's Island/Ellis Island/Liberty Island,0.0,0.0,NaN
222377,105,Manhattan,Governor's Island/Ellis Island/Liberty Island,0.0,0.0,NaN


In [28]:
# Inspect Q1 Yellow Taxi demand for the affected zones before making any treatment decision.
missing_zone_summary = (
    master.loc[
        master["zero_vehicle_household_rate"].isna()
    ]
    .groupby(
        ["LocationID", "borough", "zone"],
        as_index=False
    )
    .agg(
        estimated_total_households=("estimated_total_households", "first"),
        estimated_zero_vehicle_households=("estimated_zero_vehicle_households", "first"),
        total_pickups_q1=("pickup_count", "sum"),
        mean_hourly_pickups=("pickup_count", "mean"),
        number_of_hours=("pickup_count", "size")
    )
    .sort_values("LocationID")
)

display(missing_zone_summary)


,LocationID,borough,zone,estimated_total_households,estimated_zero_vehicle_households,total_pickups_q1,mean_hourly_pickups,number_of_hours
0,103,Manhattan,Governor's Island/Ellis Island/Liberty Island,0.0,0.0,0,0.0,2159
1,104,Manhattan,Governor's Island/Ellis Island/Liberty Island,0.0,0.0,0,0.0,2159
2,105,Manhattan,Governor's Island/Ellis Island/Liberty Island,0.0,0.0,0,0.0,2159


#### 9.3 Exclusion of the Three Non-Analysable Zones

Removal of the identified zones after the preceding inspection establishes their unavailable Census rate and zero observed Q1 Yellow Taxi demand.

In [29]:
missing_zone_ids = census_missing_zones["LocationID"].tolist()

rows_before_treatment = len(master)
zones_before_treatment = master["LocationID"].nunique()
pickup_total_before_treatment = master["pickup_count"].sum()

master_analysis = master.loc[
    ~master["LocationID"].isin(missing_zone_ids)
].copy()

print("Excluded LocationIDs:", missing_zone_ids)
print("Rows before treatment:", rows_before_treatment)
print("Rows after treatment:", len(master_analysis))
print("Rows removed:", rows_before_treatment - len(master_analysis))
print("Zones before treatment:", zones_before_treatment)
print("Zones after treatment:", master_analysis["LocationID"].nunique())
print("Remaining missing Census rate:",
      master_analysis["zero_vehicle_household_rate"].isna().sum())


Excluded LocationIDs: [103, 104, 105]
Rows before treatment: 565658
Rows after treatment: 559181
Rows removed: 6477
Zones before treatment: 262
Zones after treatment: 259
Remaining missing Census rate: 0


## 10. Data Selection for the Analysis-Ready Master Dataset

The integrated table still contains temporary join fields, preprocessing indicators, and detailed metadata that are no longer required for the next stages. This step retains the variables needed for Data Understanding, baseline and enriched modelling, zone-level evaluation, Census uncertainty assessment, and dashboard development.

This is a **column selection step only**. Variables are not removed based on model performance.


### 10.1 Standardisation of the Final Time Field

Retention of the standardised datetime values under the original `pickup_hour` variable name.

In [30]:
# Keep the standardised datetime values under the original pickup_hour column name.
print("pickup_hour dtype before:", master_analysis["pickup_hour"].dtype)
print("join_hour dtype:", master_analysis["join_hour"].dtype)

master_analysis["pickup_hour"] = master_analysis["join_hour"]

print("pickup_hour dtype after:", master_analysis["pickup_hour"].dtype)


pickup_hour dtype before: object
join_hour dtype: datetime64[ns]
pickup_hour dtype after: datetime64[ns]


### 10.2 Selection of Retained Variables

Selection of the variables required for Data Understanding, modelling, zone-level evaluation, Census uncertainty assessment, and dashboard development.

In [31]:
# Retain the variables required for the next stages of the project.
columns_to_retain = [
    "LocationID",
    "borough",
    "zone",
    "service_zone",
    "pickup_hour",
    "hour",
    "day_of_week",
    "month",
    "pickup_count",
    "temperature_f",
    "precipitation_in",
    "subway_proximity_km",
    "estimated_total_households",
    "estimated_zero_vehicle_households",
    "zero_vehicle_household_rate",
    "zero_vehicle_rate_moe",
    "zero_vehicle_rate_cv",
    "census_reliability",
    "census_coverage_ratio",
]

master_selected = master_analysis[columns_to_retain].copy()

print("Columns before selection:", master_analysis.shape[1])
print("Columns after selection:", master_selected.shape[1])


Columns before selection: 34
Columns after selection: 19


In [32]:
# Display the columns removed from the integrated dataset.
removed_columns = [
    column for column in master_analysis.columns
    if column not in master_selected.columns
]

print("Removed columns:", len(removed_columns))
for column in removed_columns:
    print("-", column)


Removed columns: 15
- date
- was_zero_padded
- join_hour
- contributing_tract_count
- census_feature_available
- census_feature_status
- nearest_station_id
- nearest_station_name
- station_borough
- nearest_station_division
- nearest_station_latitude
- nearest_station_longitude
- temperature_source
- precipitation_source
- temperature_was_interpolated


### 10.3 Final Analysis-Ready Dataset

Preview of the final selected dataset prepared for the next Data Understanding stage.

In [33]:
# Preview the final analysis-ready master dataset.
print("Final analysis-ready dataset shape:", master_selected.shape)
print("\nFinal columns:")
print(master_selected.columns.tolist())

display(master_selected.head())


Final analysis-ready dataset shape: (559181, 19)

Final columns:
['LocationID', 'borough', 'zone', 'service_zone', 'pickup_hour', 'hour', 'day_of_week', 'month', 'pickup_count', 'temperature_f', 'precipitation_in', 'subway_proximity_km', 'estimated_total_households', 'estimated_zero_vehicle_households', 'zero_vehicle_household_rate', 'zero_vehicle_rate_moe', 'zero_vehicle_rate_cv', 'census_reliability', 'census_coverage_ratio']


,LocationID,borough,zone,service_zone,pickup_hour,hour,day_of_week,month,pickup_count,temperature_f,precipitation_in,subway_proximity_km,estimated_total_households,estimated_zero_vehicle_households,zero_vehicle_household_rate,zero_vehicle_rate_moe,zero_vehicle_rate_cv,census_reliability,census_coverage_ratio
0,2,Queens,Jamaica Bay,Boro Zone,2025-01-01 00:00:00,0,2,1,0,44.96,0.0,1.598622,17.772228,2.108158,0.118621,1.106223,5.669621,low,1.0
1,2,Queens,Jamaica Bay,Boro Zone,2025-01-01 01:00:00,1,2,1,0,44.96,0.0,1.598622,17.772228,2.108158,0.118621,1.106223,5.669621,low,1.0
2,2,Queens,Jamaica Bay,Boro Zone,2025-01-01 02:00:00,2,2,1,0,44.96,0.0,1.598622,17.772228,2.108158,0.118621,1.106223,5.669621,low,1.0
3,2,Queens,Jamaica Bay,Boro Zone,2025-01-01 03:00:00,3,2,1,0,46.94,0.0,1.598622,17.772228,2.108158,0.118621,1.106223,5.669621,low,1.0
4,2,Queens,Jamaica Bay,Boro Zone,2025-01-01 04:00:00,4,2,1,0,48.02,0.0,1.598622,17.772228,2.108158,0.118621,1.106223,5.669621,low,1.0


## 11. Export the Analysis-Ready Master Dataset


In [34]:
master_selected.to_csv(OUTPUT_FILE, index=False)

print("Saved analysis-ready master dataset to:")
print(OUTPUT_FILE)
print("Saved rows:", len(master_selected))
print("Saved columns:", master_selected.shape[1])


Saved analysis-ready master dataset to:
C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs\model_datasets\nyc_yellow_taxi_master_analysis_ready_2025_q1.csv
Saved rows: 559181
Saved columns: 19


## 12. Integration Complete

The four preprocessed datasets were first integrated and validated using the complete 262-zone taxi backbone. The integrated master dataset was then inspected, and the three zones with unavailable Census zero-vehicle household rates were investigated before treatment. After confirming that these zones had no estimated households and no Yellow Taxi pickups during Q1 2025, they were excluded without imputing an artificial Census value.

The final analysis-ready dataset contains 259 taxi zones, 2,159 hourly timestamps, and 19 retained variables. Detailed distribution analysis, Census uncertainty analysis, demand-pattern analysis, low-demand-zone definition, and final model feature preparation will be performed in the Data Understanding stage.
